### Links and Resources
| Function | Use |
|---|---|
| [`concat`, `concat_ws`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.concat_ws.html) | Combine strings with/without a separator |
| [`substring`, `substr`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.substring.html) | Extract a slice |
| [`regexp_replace`, `regexp_extract`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.regexp_replace.html) | Regex on strings |
| [`split`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.split.html) | String → array |
| [`format_string`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.format_string.html) | `printf`-style formatting |

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

from pyspark.sql import functions as F

# String Functions

Strings are messy. Real-world data has trailing whitespace, mixed casing, embedded delimiters, and free-text fields begging to be parsed. Spark's string functions cover almost every cleanup you'd reach for in pandas — but vectorized, parallel, and Catalyst-aware.

In [ ]:
df = spark.createDataFrame([
    (1, "  Alice  ", "cooper", "alice@example.COM",  "Brussels, BE"),
    (2, "bob ",         "smith", "BOB@example.com",    "Paris, FR"),
    (3, "CHARLIE",           "brown", "charlie@x.io",       "Berlin, DE"),
    (4, "Dana",      "o'neill", "dana@example.org",   "Madrid, ES"),
    (5, "Darko ", "martinovic", "darko.martinovic@outlook.com", "Zagreb, HR")
], ["id", "name", "surname", "email", "location"])
display(df)

## Case and whitespace — `lower`, `upper`, `trim`, `ltrim`, `rtrim`, `initcap`

The first thing you do with any string column.

In [ ]:
cleaned = (df
    .withColumn("name_clean", F.initcap(F.trim("name")))
    .withColumn("surname_clean", F.initcap(F.trim("surname")))
    .withColumn("email_norm", F.lower(F.trim("email"))))
display(cleaned.select("id", "name_clean", "surname_clean", "email_norm"))

## Length and padding — `length`, `lpad`, `rpad`

`lpad` is how you build fixed-width identifiers (think: `"00042"`).

In [ ]:
df.select(
    "name",
    F.length("name").alias("len"),
    F.lpad(F.col("id").cast("string"), 5, "0").alias("padded_id"),
).show()

## Slicing — `substring(col, pos, len)` and `substr`

**1-indexed**, like SQL. Negative positions count from the end.

In [ ]:
df.select(
    "email",
    F.substring("email", 1, 3).alias("first3"),
    F.substring_index("email", "@", 1).alias("local_part"),
    F.substring_index("email", "@", -1).alias("domain"),
).show(truncate=False)

## Concatenation — `concat`, `concat_ws`

`concat` returns NULL if any argument is NULL. `concat_ws` (with-separator) skips NULLs — usually what you want.

In [ ]:
df.select(
    F.concat(F.lit("user_"), F.col("id")).alias("user_key"),
    F.concat_ws(" | ", "name", "email", "location").alias("summary"),
).show(truncate=False)

## Splitting — `split` returns an array

Combine with `getItem(i)` or `[i]` to pull a piece, or `explode` (notebook 06) to fan out rows.

In [ ]:
df.select(
    "location",
    F.split("location", ", ").alias("parts"),
    F.split("location", ", ").getItem(0).alias("city"),
    F.split("location", ", ").getItem(1).alias("country"),
).show(truncate=False)

## Replace and regex — `regexp_replace`, `regexp_extract`

Spark uses Java regex syntax. `regexp_extract(col, pattern, group_idx)` returns the captured group, `""` if no match.

In [ ]:
df.select(
    "email",
    F.regexp_replace("email", r"@.*$", "@REDACTED").alias("masked"),
    F.regexp_extract("email", r"^([^@]+)@(.+)$", 2).alias("domain"),
).show(truncate=False)

## Contains, startswith, endswith, like

These come back as a Boolean column — perfect for `.filter(...)`.

In [ ]:
df.filter(F.col("email").endswith(".com")) \
  .select("name", "email") \
  .show(truncate=False)

df.filter(F.col("name").rlike(r"(?i)^char")) \
  .show(truncate=False)

## `format_string` — `printf` for columns

Works like C's `printf`. Useful for building human-readable IDs.

In [ ]:
df.select(
    F.format_string("%05d - %s", "id", "name").alias("label")
).show(truncate=False)

## Takeaways

- `lower(trim(...))` is the cheapest data-cleaning move you'll do.
- Substring positions are **1-indexed**.
- Prefer `concat_ws` over `concat` — it doesn't poison the result with NULLs.
- `regexp_*` functions are full Java regex — use `(?i)` for case-insensitive.
- `split` returns an array; pull elements with `.getItem(i)`.